<a href="https://colab.research.google.com/github/21centjoe/1D-synthetic-logic-gates/blob/main/1d.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Implementation of Soliton State Checkpointing and Blockchain Verification

Here's a Python class that implements the automated checkpointing (soliton) and generates a linked hash sequence (blockchain) as per your design. This class will serialize model states, calculate their SHA-256 hashes, and maintain a JSON-based ledger for verification.

**Note:** This implementation assumes you have an external storage mounted and accessible at a specified path (e.g., `/mnt/external_drive`). Please adjust the `base_path` accordingly.

In [1]:
import os
import json
import hashlib
import datetime
import torch

class SolitonBlockchainManager:
    def __init__(self, base_path, ledger_filename='soliton_ledger.json'):
        self.base_path = base_path
        self.ledger_file = os.path.join(base_path, ledger_filename)
        os.makedirs(self.base_path, exist_ok=True)
        self.ledger = self._load_ledger()

    def _load_ledger(self):
        if os.path.exists(self.ledger_file):
            with open(self.ledger_file, 'r') as f:
                return json.load(f)
        return []

    def _save_ledger(self):
        with open(self.ledger_file, 'w') as f:
            json.dump(self.ledger, f, indent=4)

    def _calculate_file_hash(self, filepath):
        hasher = hashlib.sha256()
        with open(filepath, 'rb') as f:
            for chunk in iter(lambda: f.read(4096), b''):
                hasher.update(chunk)
        return hasher.hexdigest()

    def checkpoint_soliton_state(self, model_state, description='Model checkpoint'):
        timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S_%f') # Added microseconds
        filename = f'soliton_state_{timestamp}.pth'
        filepath = os.path.join(self.base_path, filename)

        # Serialize the model state (soliton)
        torch.save(model_state, filepath)
        print(f"Soliton state saved to: {filepath}")

        # Calculate hash of the saved file
        current_hash = self._calculate_file_hash(filepath)
        print(f"Calculated hash for current state: {current_hash}")

        # Get previous hash for linking
        previous_hash = self.ledger[-1]['hash'] if self.ledger else '0' * 64 # Genesis hash

        # Create new ledger entry
        entry = {
            'timestamp': timestamp,
            'filename': filename,
            'previous_hash': previous_hash,
            'hash': current_hash,
            'description': description
        }
        self.ledger.append(entry)
        self._save_ledger()
        print("Ledger updated with new entry.")
        return current_hash

    def verify_blockchain_integrity(self):
        print("\nVerifying blockchain integrity...")
        if not self.ledger:
            print("Ledger is empty. Nothing to verify.")
            return True

        is_intact = True
        for i, entry in enumerate(self.ledger):
            # Check if previous hash matches actual previous entry's hash
            expected_previous_hash = self.ledger[i-1]['hash'] if i > 0 else '0' * 64
            if entry['previous_hash'] != expected_previous_hash:
                print(f"[ERROR] Chain break detected at entry {i} (timestamp: {entry['timestamp']}).")
                print(f"  Expected previous hash: {expected_previous_hash}")
                print(f"  Recorded previous hash: {entry['previous_hash']}")
                is_intact = False

            # Verify the current state file's hash
            filepath = os.path.join(self.base_path, entry['filename'])
            if not os.path.exists(filepath):
                print(f"[ERROR] Soliton state file not found: {filepath}")
                is_intact = False
                continue

            actual_current_hash = self._calculate_file_hash(filepath)
            if entry['hash'] != actual_current_hash:
                print(f"[ERROR] Data tampering detected in file: {filepath}")
                print(f"  Recorded hash: {entry['hash']}")
                print(f"  Calculated hash: {actual_current_hash}")
                is_intact = False

        if is_intact:
            print("Blockchain integrity verified: All states and links are intact.")
        else:
            print("Blockchain integrity verification FAILED.")
        return is_intact

    def get_ledger(self):
        return self.ledger

### Demonstration of the SolitonBlockchainManager

Let's simulate a few model checkpoints and then verify the integrity of the blockchain.

In [2]:
# Define your external drive path. Replace with your actual mounted path.
# Example for Google Drive: /content/drive/MyDrive/colab_checkpoints
# Example for a specific mounted external drive: /mnt/external_drive/soliton_data
external_drive_path = './soliton_checkpoints'

# Initialize the manager
manager = SolitonBlockchainManager(base_path=external_drive_path)

# Simulate some model states (e.g., PyTorch model's state_dict)
# In a real scenario, this would be your actual model's state.
model_state_1 = {'layer1.weight': torch.randn(10, 5), 'layer1.bias': torch.randn(10)}
model_state_2 = {'layer1.weight': torch.randn(10, 5) * 1.01, 'layer1.bias': torch.randn(10)}
model_state_3 = {'layer1.weight': torch.randn(10, 5) * 0.99, 'layer1.bias': torch.randn(10)}

print("\n--- Checkpointing State 1 ---")
manager.checkpoint_soliton_state(model_state_1, description='Epoch 10: Model after initial training')

print("\n--- Checkpointing State 2 ---")
manager.checkpoint_soliton_state(model_state_2, description='Epoch 20: Model after fine-tuning')

print("\n--- Checkpointing State 3 ---")
manager.checkpoint_soliton_state(model_state_3, description='Epoch 30: Model after regularization')

# Display the full ledger
print("\n--- Current Blockchain Ledger ---")
display(manager.get_ledger())

# Verify the integrity
manager.verify_blockchain_integrity()

print("\n--- Simulating tampering (optional) ---")
# To simulate tampering, you could manually modify a file in the 'soliton_checkpoints' folder
# or manually edit the 'soliton_ledger.json' file and then run verify_blockchain_integrity() again.
# Example: os.system(f"echo 'tampered data' >> {os.path.join(external_drive_path, manager.ledger[0]['filename'])}")
# manager.verify_blockchain_integrity()


NameError: name 'SolitonBlockchainManager' is not defined

In [ ]:
import shutil
import os

# Define the directory to clear
checkpoints_dir = './soliton_checkpoints'

# Check if the directory exists and remove it
if os.path.exists(checkpoints_dir):
    shutil.rmtree(checkpoints_dir)
    print(f"Cleared directory: {checkpoints_dir}")
else:
    print(f"Directory not found: {checkpoints_dir} (nothing to clear)")

# Recreate the base path, as the SolitonBlockchainManager expects it to exist or create it.
os.makedirs(checkpoints_dir, exist_ok=True)
